In [ ]:
# Import Python libraries
import sqlite3
import pandas as pd

# Connect to SQlite database and load tweets table
conn = sqlite3.connect('database.sqlite')
df = pd.read_sql_query("SELECT * FROM Tweets", conn)
conn.close()

# Preview first 5 rows
df.head()

In [5]:
# Checking the fullsize of the dataset
print(df.shape)

(14485, 15)


In [6]:
# Checking for missing values
df.isnull().sum()

tweet_id                        0
airline_sentiment               0
airline_sentiment_confidence    0
negativereason                  0
negativereason_confidence       0
airline                         0
airline_sentiment_gold          0
name                            0
negativereason_gold             0
retweet_count                   0
text                            0
tweet_coord                     0
tweet_created                   0
tweet_location                  0
user_timezone                   0
dtype: int64

In [15]:
# Keep only columns relevant to sentiment analysis
## We reduce the dataset to 7 relevant columns and clean the tweet text before running sentiment scoring
columns_to_keep = [
    'text',
    'airline_sentiment',
    'airline_sentiment_confidence',
    'negativereason',
    'airline',
    'retweet_count',
    'tweet_created'
]

df = df[columns_to_keep].copy()
df.shape

(14485, 7)

In [16]:
df.head()

,text,airline_sentiment,airline_sentiment_confidence,negativereason,airline,retweet_count,tweet_created
0,@JetBlue's new CEO seeks the right balance to ...,neutral,1.0,,Delta,0,2015-02-16 23:36:05 -0800
1,@JetBlue is REALLY getting on my nerves !! 😡😡 ...,negative,1.0,Can't Tell,Delta,0,2015-02-16 23:43:02 -0800
2,@united yes. We waited in line for almost an h...,negative,1.0,Late Flight,United,0,2015-02-16 23:48:48 -0800
3,@united the we got into the gate at IAH on tim...,negative,1.0,Late Flight,United,0,2015-02-16 23:52:20 -0800
4,@SouthwestAir its cool that my bags take a bit...,negative,1.0,Customer Service Issue,Southwest,0,2015-02-17 00:00:36 -0800


In [17]:
# Import cleaning library
import re

In [18]:
# Function to clean raw tweet text
def clean_tweet(text):
    text = re.sub(r'http\S+', '', text) # remove URLs
    text = re.sub(r'@\w+', '', text) # remove @mentions
    text = re.sub(r'#\w+', '', text) # remove Hashtags
    text = re.sub(r'[^A-Za-z\s]', '', text) # remove Special characters
    text = text.lower().strip() # remove Lowercase and trimcase
    return text
    

In [19]:
# Apply cleaning function to every tweet
df['clean_text'] = df['text'].apply(clean_tweet)

#Compare original vs cleaned
df[['text', 'clean_text']].head()

,text,clean_text
0,@JetBlue's new CEO seeks the right balance to ...,s new ceo seeks the right balance to please pa...
1,@JetBlue is REALLY getting on my nerves !! 😡😡 ...,is really getting on my nerves
2,@united yes. We waited in line for almost an h...,yes we waited in line for almost an hour to do...
3,@united the we got into the gate at IAH on tim...,the we got into the gate at iah on time and ha...
4,@SouthwestAir its cool that my bags take a bit...,its cool that my bags take a bit longer dont g...


In [20]:
## Sentiment scoring with VADER 
### VADER works by reading each word in a tweet and assigning it a score. It then combines everything into one final score between -1 & +1, and labels it positive, negative or neutral. This make the analysis quantitative and not just descriptive.

In [22]:
!pip install vaderSentiment


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

# Initialize the analyzer
analyzer = SentimentIntensityAnalyzer()

In [24]:
# Function to extract the compound sentiment score
def get_sentiment_score(text):
    score = analyzer.polarity_scores(text)
    return score['compound']

# Apply to clean_text column
df['sentiment_score'] = df['clean_text'].apply(get_sentiment_score)

df[['clean_text', 'sentiment_score']].head()

,clean_text,sentiment_score
0,s new ceo seeks the right balance to please pa...,0.3182
1,is really getting on my nerves,-0.1027
2,yes we waited in line for almost an hour to do...,0.4019
3,the we got into the gate at iah on time and ha...,0.0000
4,its cool that my bags take a bit longer dont g...,0.3182


In [25]:
# Classify score into positive, negative or neutral
def classify_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['vader_sentiment'] = df['sentiment_score'].apply(classify_sentiment)

df[['clean_text', 'sentiment_score', 'vader_sentiment']].head()

,clean_text,sentiment_score,vader_sentiment
0,s new ceo seeks the right balance to please pa...,0.3182,positive
1,is really getting on my nerves,-0.1027,negative
2,yes we waited in line for almost an hour to do...,0.4019,positive
3,the we got into the gate at iah on time and ha...,0.0000,neutral
4,its cool that my bags take a bit longer dont g...,0.3182,positive


In [29]:
## scikit-learn is the go-to Python library for machine learning and evaluation metrics like accuracy scores and classification reports.
!pip install scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
## To comfirm if my sentiment analysis is correct
# I compare VADER predictions against the original dataset labels
from sklearn.metrics import classification_report, accuracy_score

# Calculate accuracy
accuracy = accuracy_score(df['airline_sentiment'], df['vader_sentiment'])
print(f"VADER Accuracy: {accuracy:.2%}")

# Detailed breakdown
print(classification_report(df['airline_sentiment'], df['vader_sentiment']))

VADER Accuracy: 54.15%
              precision    recall  f1-score   support

    negative       0.90      0.50      0.64      9082
     neutral       0.38      0.43      0.40      3069
    positive       0.34      0.87      0.48      2334

    accuracy                           0.54     14485
   macro avg       0.54      0.60      0.51     14485
weighted avg       0.70      0.54      0.56     14485



In [30]:
# Original labels distribution
print("Original sentiment distribution:")
print(df['airline_sentiment'].value_counts())

print("\nVADER sentiment distribution:")
print(df['vader_sentiment'].value_counts())

Original sentiment distribution:
airline_sentiment
negative    9082
neutral     3069
positive    2334
Name: count, dtype: int64

VADER sentiment distribution:
vader_sentiment
positive    6015
negative    5037
neutral     3433
Name: count, dtype: int64


*VADER achieved 54% accuracy on informal tweet data. This is consistent with literature because tweets contain sarcasm, slang, and abbreviations that challenge rule based sentiment scorers. The original labels were human annonated, giving VADER a high quality benchmark to compare against*
*VADER tends to interpret neutral or mildy negative tweets as positive tweets because it scores words in isolation without understanding context or sarcasm*